# unit08 レッスン: 画像とパターン認識の基礎

**題材** — 出品写真801枚から `plate / book / bottle / shirt / shoe` の5クラスを当て、
同一商品の「少しだけ違う写真」も見つける。画像は外部ダウンロードなしの64×64 RGB PNG。

今日の中心はひとつだけ:

> **画像は `(高さ, 幅, チャネル)` の数値配列。既習の NumPy がそのまま武器になる。**

## このレッスンを終えると作れるようになるもの

1. HWC / CHW、RGB / BGR、uint8 / float32 を区別し、shape・dtype・値域を壊さず変換できる
2. リサイズ、グレースケール、チャネル別標準化を shape と axis で説明して実装できる
3. 完全一致で拾えない近傍重複を、知覚ハッシュと正規化画像の類似度で検出できる
4. 色ヒストグラムと HOG / 輪郭特徴を作り、`product_key` 単位のCVで比較できる
5. ラベルを保つ拡張を学習foldだけへ適用し、誤分類を画像IDまで戻して調査できる

所要の目安: **20〜25分**。このあと演習 `ex01`〜`ex04` が続く。

各概念を **見る → 予測する → 変える → 書く → チェック** の順で進む。
未記入でも notebook は止まらず、チェックポイントが `[NG]` と修正の方向を返す。

In [ ]:
# STEP 1: このレッスンを終えると作れるようになるものの処理を実行し、出力を照合する
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from skimage import filters
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

DATA = Path("data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit08-image-and-pattern-basics/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"
assert (DATA / "images" / "train").exists(), f"画像ディレクトリが見つかりません: {DATA.resolve()}"

train = pd.read_csv(DATA / "train.csv")
print("train:", train.shape, "/ 商品数:", train["product_key"].nunique())
print("DATA =", DATA.resolve())


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def call_safely(fn, *args, **kwargs):
    # 未完成の関数を呼んでも notebook を止めない。
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数内で {type(e).__name__}: {e})")
        return None


def shape_safely(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None


def item_safely(value, index):
    try:
        return value[index]
    except Exception:
        return None


print("セットアップ完了。ヘルパー: check / call_safely / shape_safely")

---
# 概念1 — 画像の配列表現: HWC / CHW、dtype、チャネル順

## ① なぜ: 見た目が同じでも、配列の規約が違えばモデルは壊れる

画像処理のバグは、アルゴリズムより **shape・dtype・値域・チャネル順**から生まれることが多い。
RGB を BGR として読むと赤と青が入れ替わり、uint8 のまま足すと255を越えた値が0付近へ巻き戻る。

新しい画像を受け取ったら、最初の3行はいつも `shape / dtype / min-max`。
unit07 の token batch と同じで、まず配列の契約を確定してから処理を書く。

## ② 解説: ライブラリごとの規約を翻訳する

| 規約 | shape / 値 | 主なライブラリ | C#での見方 |
|---|---|---|---|
| HWC | `(height, width, channels)` | Pillow / scikit-image / OpenCV | `Color[height,width]` の各要素に3成分 |
| CHW | `(channels, height, width)` | PyTorch | RGB 3枚の `float[,]` を積む |
| RGB | 末尾/先頭チャネルが Red, Green, Blue | Pillow / skimage / PyTorch | 通常の `Color.R/G/B` |
| BGR | Blue, Green, Red | **OpenCVだけ** | 読込後 `[..., ::-1]` でRGBへ |
| uint8 | 0〜255 の整数 | PNG/JPEGの読込直後 | `byte`。255+1は256にならず巻き戻る |
| float32 | 通常0.0〜1.0 | 学習・数値演算 | `(float)pixel / 255f` |

`np.transpose(img, (2, 0, 1))` は軸順を HWC→CHW に並べ替える。データはコピーせず view のこともある。
RGBA は4チャネル目が alpha(透過度)。RGBモデルへ渡すなら背景と合成して3チャネルへする。

In [ ]:
# GOAL: 実画像を読み、HWC uint8 → CHW float32 の各段階で契約を確認する

first_path = DATA / "images" / "train" / train["image_id"].iat[0]
with Image.open(first_path) as im:
    image_hwc = np.asarray(im.convert("RGB"))

print("読込直後:", image_hwc.shape, image_hwc.dtype,
      "値域=", int(image_hwc.min()), "-", int(image_hwc.max()))

image_float = image_hwc.astype(np.float32) / 255.0
print("float化  :", image_float.shape, image_float.dtype,
      "値域=", round(float(image_float.min()), 4), "-", round(float(image_float.max()), 4))

image_chw = np.transpose(image_float, (2, 0, 1))
print("CHW化    :", image_chw.shape, image_chw.dtype)

# uint8 の危険を極小値で確認。
u = np.array([250], dtype=np.uint8)
print("\nuint8 250 + 10 =", int((u + 10)[0]), "← 260ではなく4へ巻き戻る")
print("float32へ変換後  =", float(u.astype(np.float32)[0] + 10))

## ④ 予測: 軸順を間違えると、どんな shape になる?

次のセルは `(H,W,C)=(2,3,3)` の極小画像で軸を追う。

1. `transpose(2,0,1)` の結果は?
2. `transpose(1,2,0)` と書き間違えると?
3. BGR→RGB の `[..., ::-1]` は shape を変える?
4. uint8 の画像へ先に `/255` すれば Python は float に昇格するが、学習用 dtype を明示する理由は?

実行前に各軸名を書き、数字だけで推測しないこと。

In [ ]:
# GOAL: transpose は「どの軸を何番目へ持ってくるか」で読む

tiny_hwc = np.array([
    [[0, 10, 20], [255, 128, 64], [30, 60, 90]],
    [[12, 24, 36], [48, 96, 144], [200, 100, 50]],
], dtype=np.uint8)
print("HWC:", tiny_hwc.shape)
print("正しい CHW transpose(2,0,1):", tiny_hwc.transpose(2, 0, 1).shape)
print("誤った軸順 transpose(1,2,0):", tiny_hwc.transpose(1, 2, 0).shape)

bgr = tiny_hwc[..., ::-1]
print("\nRGB先頭pixel:", tiny_hwc[0, 0].tolist())
print("BGR先頭pixel:", bgr[0, 0].tolist())
print("shape は同じ:", bgr.shape)

## ⑥ 書いてみる: HWC uint8 を CHW float32 にする

`to_chw_float(image)` を完成させよう。

- `np.asarray(image)` で配列として受け取る
- `astype(np.float32) / 255.0` で値域を0〜1へ
- `np.transpose(..., (2,0,1))` で CHW へ
- 元の配列を書き換えない

3行で書ける。変換後も `shape / dtype / min-max` を確認する癖をつける。

In [ ]:
# STEP 4: ⑥ 書いてみる: HWC uint8 を CHW float32 にするの処理を実行し、出力を照合する
def to_chw_float(image):
    # ここに書く(ヒント: float32化して255で割り、軸を (2,0,1) に並べ替える)
    return None


chw_a = call_safely(to_chw_float, tiny_hwc)
print("chw_a:", shape_safely(chw_a), None if chw_a is None else chw_a.dtype)

In [ ]:
# STEP 5: ⑥ 書いてみる: HWC uint8 を CHW float32 にするの処理を実行し、出力を照合する
# ===== チェックポイント A: HWC → CHW =====
check("A-1 shape", shape_safely(chw_a), (3, 2, 3),
      hint="元は (H,W,C)=(2,3,3)。Cを先頭へ。")
check("A-2 dtype", str(chw_a.dtype) if chw_a is not None else None, "float32",
      hint="astype(np.float32) を明示する。")
_a_value = float(chw_a[0, 0, 1]) if chw_a is not None else None
check("A-3 値域変換", _a_value, 1.0,
      hint="元の R=255 を255で割る。")
_a_input = int(tiny_hwc[0, 1, 0]) if chw_a is not None else None
check("A-4 入力を破壊しない", _a_input, 255,
      hint="astype は新しい配列を返す。元の uint8 配列へ代入しない。")

---
# 概念2 — リサイズ、グレースケール、チャネル別標準化

## ① なぜ: 入力の大きさと明るさをそろえて、形の比較を公平にする

出品写真は解像度・余白・照明がばらばら。同じ商品でも、画素数や背景の明るさだけで距離が変わる。
モデルへ入れる前にサイズと値の尺度をそろえると、「明るい写真か」ではなく「どんな形か」を学びやすくなる。

ただし、縦横比を無視した resize は形そのものを潰す。分類対象に形が重要なら、比率を保って余白を足す設計も比較する。

## ② 解説: axis を名前で読む

| 処理 / API | 何をするものか | 注意 |
|---|---|---|
| `PIL.Image.resize((W,H), resample=...)` | 画像を指定サイズへ補間する | 引数順が `(幅,高さ)`。配列 shape の `(H,W)` と逆 |
| NEAREST | 最も近い1画素を使う | マスク・ドット絵向け。写真ではギザギザ |
| BILINEAR | 周囲4画素を線形補間 | 写真の軽いresizeの出発点 |
| grayscale | RGBの輝度重み付き和 | `0.2126R + 0.7152G + 0.0722B` |
| `mean(axis=(0,1), keepdims=True)` | H,Wを集計しチャネルだけ残す | shape `(1,1,3)`。HWC画像へブロードキャスト可能 |
| `(img-mean)/std` | チャネルごとに中心0・分散1へ | std=0 のチャネルには保護が必要 |

`axis=(0,1)` は高さと幅。`axis=2` はチャネルを潰してグレースケールを作る方向。
数字を暗記せず、毎回 `print(img.shape)` して軸名を書き添える。

In [ ]:
# GOAL: resize → grayscale → channel標準化を、各段階の shape と値域で追う

with Image.open(first_path) as im:
    rgb_pil = im.convert("RGB")
    resized_pil = rgb_pil.resize((32, 32), resample=Image.Resampling.BILINEAR)
resized = np.asarray(resized_pil, dtype=np.float32) / 255.0
print("resize後:", resized.shape, resized.dtype,
      "値域=", round(float(resized.min()), 3), "-", round(float(resized.max()), 3))

gray = resized @ np.array([0.2126, 0.7152, 0.0722], dtype=np.float32)
print("gray:", gray.shape, "← channel軸が消えた")

channel_mean = resized.mean(axis=(0, 1), keepdims=True)
channel_std = resized.std(axis=(0, 1), keepdims=True)
standardized = (resized - channel_mean) / np.maximum(channel_std, 1e-6)
print("mean/std shape:", channel_mean.shape, channel_std.shape)
print("標準化後のchannel平均:", np.round(standardized.mean(axis=(0, 1)), 6))
print("標準化後のchannel標準偏差:", np.round(standardized.std(axis=(0, 1)), 6))

## ④ 予測: `keepdims=False` でも動くのはなぜ? いつ危ない?

HWC画像なら `mean(axis=(0,1))` は shape `(3,)` で、末尾チャネルへ偶然きれいにブロードキャストできる。

1. CHW画像 `(3,H,W)` へ `(3,)` を引ける?
2. HWCで `keepdims=True` の `(1,1,3)` は、何を集計したか shape だけで読める?
3. RGB 全チャネルを一括で1つの平均へ潰すと、色かぶりの補正はどう変わる?

`keepdims=True` は「たまたま動く」から「意図がshapeに残る」へ変える保険だ。

In [ ]:
# GOAL: HWCとCHWで、チャネル統計を当てるshapeが違うことを見る

tiny_float = tiny_hwc.astype(np.float32) / 255.0
mean_hwc = tiny_float.mean(axis=(0, 1), keepdims=True)
chw_float = tiny_float.transpose(2, 0, 1)
mean_chw = chw_float.mean(axis=(1, 2), keepdims=True)

print("HWC image / mean:", tiny_float.shape, mean_hwc.shape)
print("CHW image / mean:", chw_float.shape, mean_chw.shape)
print("HWC標準化の式: (H,W,C) - (1,1,C)")
print("CHW標準化の式: (C,H,W) - (C,1,1)")

nearest = np.asarray(Image.fromarray(tiny_hwc).resize((9, 6), Image.Resampling.NEAREST))
bilinear = np.asarray(Image.fromarray(tiny_hwc).resize((9, 6), Image.Resampling.BILINEAR))
print("\nresize shapeは同じ:", nearest.shape, bilinear.shape)
print("補間結果が違う画素数:", int(np.any(nearest != bilinear, axis=2).sum()))

## ⑥ 書いてみる: HWC画像をチャネル別に標準化する

`standardize_channels(image)` を完成させ、`(normalized, mean, std)` を返そう。

- float配列へ変換
- mean / std は `axis=(0,1), keepdims=True`
- 0除算を避けるため分母は `np.maximum(std, 1e-6)`
- 元画像と同じ shape の normalized を返す

4行で書ける。shape を先に印字してから式を書こう。

In [ ]:
# STEP 8: ⑥ 書いてみる: HWC画像をチャネル別に標準化するの処理を実行し、出力を照合する
def standardize_channels(image):
    # ここに書く(ヒント: H,Wを集計し、C軸を残した (1,1,C) のmean/stdを作る)
    return None


result_b = call_safely(standardize_channels, np.arange(24, dtype=float).reshape(2, 4, 3) / 23.0)
normalized_b = item_safely(result_b, 0)
mean_b = item_safely(result_b, 1)
std_b = item_safely(result_b, 2)
print("normalized:", shape_safely(normalized_b), "mean:", shape_safely(mean_b), "std:", shape_safely(std_b))

In [ ]:
# STEP 9: ⑥ 書いてみる: HWC画像をチャネル別に標準化するの処理を実行し、出力を照合する
# ===== チェックポイント B: チャネル別標準化 =====
check("B-1 normalized shape", shape_safely(normalized_b), (2, 4, 3),
      hint="標準化は要素数を変えない。")
check("B-2 mean shape", shape_safely(mean_b), (1, 1, 3),
      hint="axis=(0,1), keepdims=True。")
check("B-3 channel平均", normalized_b.mean(axis=(0, 1)) if normalized_b is not None else None,
      [0.0, 0.0, 0.0], hint="各channelから、そのchannel自身のmeanを引く。")
check("B-4 channel標準偏差", normalized_b.std(axis=(0, 1)) if normalized_b is not None else None,
      [1.0, 1.0, 1.0], hint="各channelを、そのchannel自身のstdで割る。")

---
# 概念3 — 近傍重複検出: 完全一致から知覚表現へ

## ① なぜ: 同じ商品でも、ピクセルはほぼ1つも一致しない

同じ写真を縮小して保存し直す、余白を足す、ロゴを焼く、明るさを変える。
人には同じ商品でも、PNGバイト列や全ピクセルは別物になる。

実測では同一商品ペア378件のうち**ピクセル完全一致は3件だけ**。
完全一致だけでは99%近くを取りこぼし、重複が train / valid に割れて検証リークを起こす。
unit06 の「ペアを作って同一判定する」が、画像でも同じ形で再登場する。

## ② 解説: 見た目を小さな表現へ落として比べる

**average hash(aHash)** は、グレースケール画像を8×8へ縮小し、平均以上を1、未満を0にして64bitへ落とす。
ハッシュ間は XOR 後の1の個数、つまりハミング距離で比較する。

| 手法 | 比較するもの | 強み / 弱み |
|---|---|---|
| バイト/完全一致 | 全ピクセルが同じか | 高速だが再保存・resize・明度で壊れる |
| aHash / dHash | 縮小後の明暗パターン | 小変化に強いが、大回転・大cropには弱い |
| 色ヒストグラム | 色の分布 | 位置に強いが、同色の別形状を混同 |
| 正規化縮小画像 cosine | 16×16の明暗配置 | このデータでは同一0.998 / 別0.372の中央値 |

NumPy のブールマスクと `!=` がそのまま64bit比較になる。実務では64個のboolを整数へpackして索引化する。

In [ ]:
# GOAL: 同一商品の変形写真が完全一致でなくても、正規化表現では近いことを見る

counts = train["product_key"].value_counts()
duplicate_key = counts[counts >= 2].index[0]
same_rows = train[train["product_key"] == duplicate_key].iloc[:2]
different_row = train[train["product_key"] != duplicate_key].iloc[0]


def load_rgb(row):
    path = DATA / "images" / "train" / row["image_id"]
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


def normalized_thumbnail(image, size=16):
    gray = Image.fromarray(image).convert("L").resize((size, size), Image.Resampling.BILINEAR)
    vector = np.asarray(gray, dtype=np.float32).reshape(-1)
    vector = vector - vector.mean()
    vector = vector / max(float(vector.std()), 1e-6)
    return vector / max(float(np.linalg.norm(vector)), 1e-6)


same_a, same_b = [load_rgb(row) for _, row in same_rows.iterrows()]
other = load_rgb(different_row)
print("同一商品でピクセル完全一致:", bool(np.array_equal(same_a, same_b)))
print("同一商品のthumbnail cosine:", round(float(normalized_thumbnail(same_a) @ normalized_thumbnail(same_b)), 4))
print("別商品のthumbnail cosine  :", round(float(normalized_thumbnail(same_a) @ normalized_thumbnail(other)), 4))
print("実測中央値は同一0.998 / 別0.372。単一例ではなく分布で閾値を決める。")

## ④ 予測: 閾値を下げると、recall と誤検出はどう動く?

正本の実測表を次のセルで表示する。閾値は0.95→0.80へ下げる。

1. 同一商品を拾う recall は上がる? 下がる?
2. 別商品を誤って同一とする誤検出率は?
3. 重複除去で「別商品を消す事故」と「重複を残す事故」のどちらが痛いかで、閾値はどう変わる?

unit06 の precision / recall とまったく同じ設計判断になる。

In [ ]:
# GOAL: 閾値を1つの正解ではなく、用途で選ぶトレードオフとして読む

duplicate_thresholds = pd.DataFrame({
    "threshold": [0.95, 0.90, 0.85, 0.80],
    "recall": [0.722, 0.733, 0.762, 0.825],
    "false_positive_rate": [0.0000, 0.0013, 0.0043, 0.0150],
})
print(duplicate_thresholds.to_string(index=False))
print("\n閾値0.95: 誤検出を避けたい自動削除向け。ただし約28%を取りこぼす。")
print("閾値0.80: recallは82.5%へ上がるが、別商品の1.5%を誤検出。人手確認候補向け。")
print("同一商品378ペア中、完全一致は3件だけ。完全一致を主役にしてはいけない。")

## ⑥ 書いてみる: 8×8グレースケールの average hash

`average_hash_bits(gray8)` を完成させよう。入力はすでに8×8へ縮小済みの2次元配列。

- 全64画素の平均を取る
- 平均以上を1、未満を0にする
- `astype(np.uint8).reshape(-1)` で長さ64の0/1配列を返す

2〜3行。2つのhashのハミング距離は `np.sum(hash_a != hash_b)` で出る。

In [ ]:
# STEP 12: ⑥ 書いてみる: 8×8グレースケールの average hashの処理を実行し、出力を照合する
def average_hash_bits(gray8):
    # ここに書く(ヒント: 平均以上かのbool配列を uint8 にし、1次元へ)
    return None


hash_c = call_safely(average_hash_bits, np.arange(64, dtype=float).reshape(8, 8))
print("hash_c:", shape_safely(hash_c), None if hash_c is None else int(hash_c.sum()))

In [ ]:
# STEP 13: ⑥ 書いてみる: 8×8グレースケールの average hashの処理を実行し、出力を照合する
# ===== チェックポイント C: average hash =====
check("C-1 shape", shape_safely(hash_c), (64,),
      hint="8×8を reshape(-1) で64要素へ。")
check("C-2 1の個数", int(hash_c.sum()) if hash_c is not None else None, 32,
      hint="0〜63の平均は31.5。32〜63の32画素が平均以上。")
check("C-3 dtype", str(hash_c.dtype) if hash_c is not None else None, "uint8",
      hint="boolのままではなく astype(np.uint8)。")
_c_reverse = call_safely(average_hash_bits, 63.0 - np.arange(64, dtype=float).reshape(8, 8))
_c_hamming = int(np.sum(hash_c != _c_reverse)) if hash_c is not None and _c_reverse is not None else None
check("C-4 反転画像とのハミング距離", _c_hamming, 64,
      hint="大小関係が全画素で反転するので、64bitすべて異なる。")

---
# 概念4 — 色ヒストグラム、HOG、輪郭と group-aware CV

## ① なぜ: 生ピクセルではなく、不変にしたいものを特徴へ入れる

生ピクセルは1画素ずれるだけで別の列へ証拠が移り、平行移動・照明・背景へ弱い。
古典特徴は「何を同じとみなすか」を人が設計する。

色ヒストグラムは位置を捨てて色分布を残す。HOG は局所勾配の向きを数え、輪郭・形を残す。
このデータは色をクラス間で共有してあるため、**色0.3533に対し HOG 0.7840**。
形を特徴へ入れる価値が数字で見える。

## ② 解説: 特徴抽出と分割はセットで設計する

| API / 特徴 | 何をするものか | 戻り値・注意 |
|---|---|---|
| `np.histogram(channel, bins, range)` | 値域を区間に分け画素数を数える | channelごとに正規化して連結 |
| `filters.sobel(gray)` | 輝度の急変=エッジ強度を出す | 2D画像。輪郭量や領域統計に使う |
| `skimage.feature.hog` | セルごとに勾配方向ヒストグラムを作り、block正規化 | `feature_vector=True` で1次元特徴 |
| `LinearSVC` | 高次元特徴に線形境界を学習する分類器 | `fit/predict`。確率は返さない |
| `StratifiedGroupKFold` | クラス比を寄せつつ group をfold間で分離 | `groups=product_key` が必須 |

同一商品には変形写真が複数ある。画像単位の StratifiedKFold では、ほぼ同じ商品が train / valid に割れてリークする。
`product_key` は同じfoldへ閉じ込める。unit02 の group leakage が画像でも再登場する。

HOGの直感: `画素 → 勾配(向き+強さ) → 8×8セルで方向を投票 → 2×2セルblockで明るさ正規化 → 連結`。

In [ ]:
# GOAL: 同じ画像から「色」「輪郭」「HOG」の3種類の特徴を作り、shapeを見る

def _color_hist_reference(image, bins=8):
    parts = []
    for channel in range(3):
        counts, _ = np.histogram(image[..., channel], bins=bins, range=(0, 256))
        parts.append(counts.astype(float) / max(1, counts.sum()))
    return np.concatenate(parts)


sample_float = image_hwc.astype(np.float32) / 255.0
sample_gray = sample_float @ np.array([0.2126, 0.7152, 0.0722])
color_feature = _color_hist_reference(image_hwc, bins=8)
edge_map = filters.sobel(sample_gray)
hog_feature = hog(sample_gray, orientations=9, pixels_per_cell=(8, 8),
                  cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True)

print("image:", image_hwc.shape)
print("color histogram:", color_feature.shape, "= 3ch × 8bins")
print("edge map:", edge_map.shape, "平均edge強度=", round(float(edge_map.mean()), 4))
print("HOG:", hog_feature.shape, "値域=", round(float(hog_feature.min()), 4), "-", round(float(hog_feature.max()), 4))

## ④ 予測: 色だけと形、どちらが5クラスを分ける?

次のセルは全801枚で特徴を1回作り、`StratifiedGroupKFold` のOOF予測を作る。

1. 色はクラス間で共有されている。chance 0.20からどれくらい上がる?
2. HOG は位置・回転へ完全に不変? ±25度や遮蔽で何が起きる?
3. 各foldの train / valid の `product_key` 積集合は何件であるべき?
4. accuracy と macro-F1 が離れたら、まず何を疑う?

正本の基準値は色0.3533、HOG 0.7840。ここでは分割方法を明示して同じ手順を再実行する。

In [ ]:
# GOAL: product単位で分けたOOFを作り、accuracy / macro-F1 / 誤分類を確認する

images_all = []
for image_id in train["image_id"]:
    with Image.open(DATA / "images" / "train" / image_id) as im:
        images_all.append(np.asarray(im.convert("RGB")))
images_all = np.stack(images_all)
print("images_all:", images_all.shape, images_all.dtype)

color_X = np.stack([_color_hist_reference(img, bins=8) for img in images_all])
gray_all = images_all.astype(np.float32) @ np.array([0.2126, 0.7152, 0.0722], dtype=np.float32) / 255.0
hog_X = np.stack([
    hog(g, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
        block_norm="L2-Hys", feature_vector=True)
    for g in gray_all
])
print("color_X:", color_X.shape, "/ hog_X:", hog_X.shape)

y = train["label"].to_numpy()
groups = train["product_key"].to_numpy()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)


def group_oof(X):
    pred = np.empty(len(y), dtype=object)
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups), start=1):
        overlap = set(groups[tr_idx]) & set(groups[va_idx])
        assert not overlap
        model = LinearSVC(C=1.0, dual="auto", max_iter=5000)
        model.fit(X[tr_idx], y[tr_idx])
        pred[va_idx] = model.predict(X[va_idx])
        print(f"fold{fold}: train={len(tr_idx)} valid={len(va_idx)} group重複={len(overlap)}")
    return pred


print("\n--- 色ヒストグラム ---")
oof_color = group_oof(color_X)
print("\n--- HOG ---")
oof_hog = group_oof(hog_X)

rows = []
for name, pred in [("色hist", oof_color), ("HOG", oof_hog)]:
    rows.append({"feature": name,
                 "accuracy": accuracy_score(y, pred),
                 "macro_f1": f1_score(y, pred, average="macro", zero_division=0)})
measured_scores = pd.DataFrame(rows)
print("\nこの StratifiedGroupKFold 実行のOOF:")
print(measured_scores.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nデータ正本の検証値: 色hist+LinearSVC=0.3533 / HOG+LinearSVC=0.7840 accuracy")

errors = train.loc[oof_hog != y, ["image_id", "product_key", "label"]].copy()
errors["predicted"] = oof_hog[oof_hog != y]
print("\nHOG誤分類:", len(errors), "件 / 先頭8件")
print(errors.head(8).to_string(index=False))
print("\nconfusion matrix (row=true, col=pred):")
labels_order = sorted(train["label"].unique())
print(pd.DataFrame(confusion_matrix(y, oof_hog, labels=labels_order),
                   index=labels_order, columns=labels_order))

## ⑥ 書いてみる: 正規化したRGB色ヒストグラム

`color_histogram_hwc(image, bins=4)` を完成させよう。

- RGB各チャネルへ `np.histogram(..., bins=bins, range=(0,256))`
- 各チャネルの counts を画素数で割り、合計1へ正規化
- 3本を `np.concatenate` して長さ `3*bins` で返す

5〜7行。色の「どこにあるか」は捨て、各色の「どれだけあるか」だけを残す特徴になる。

In [ ]:
# STEP 16: ⑥ 書いてみる: 正規化したRGB色ヒストグラムの処理を実行し、出力を照合する
def color_histogram_hwc(image, bins=4):
    # ここに書く(ヒント: channelごとにhistogram→合計1へ正規化→3本を連結)
    return None


hist_d = call_safely(color_histogram_hwc, np.array([
    [[0, 0, 10], [0, 64, 20]], [[255, 128, 30], [255, 255, 40]]
], dtype=np.uint8), 4)
print("hist_d:", shape_safely(hist_d), hist_d)

In [ ]:
# STEP 17: ⑥ 書いてみる: 正規化したRGB色ヒストグラムの処理を実行し、出力を照合する
# ===== チェックポイント D: 色ヒストグラム =====
check("D-1 shape", shape_safely(hist_d), (12,),
      hint="RGB 3ch × 4bins。")
check("D-2 値", hist_d, [0.5, 0.0, 0.0, 0.5, 0.25, 0.25, 0.25, 0.25, 1.0, 0.0, 0.0, 0.0],
      hint="range=(0,256) を明示し、channelごとに画素数4で割る。")
_d_sums = np.array([hist_d[i*4:(i+1)*4].sum() for i in range(3)]) if hist_d is not None else None
check("D-3 channelごとの合計", _d_sums, [1.0, 1.0, 1.0],
      hint="3ch全体で1ではなく、各channelが1。")
check("D-4 入力shapeに依存しない長さ",
      shape_safely(call_safely(color_histogram_hwc, np.zeros((8, 5, 3), dtype=np.uint8), 4)), (12,),
      hint="画素数が変わっても特徴長は 3*bins。")

---
# 概念5 — データ拡張と誤分類分析

## ① なぜ: 「変わっても同じクラス」を学習foldで教える

位置、明るさ、小さな回転、撮影ノイズは、商品のクラスを変えない nuisance factor。
データ拡張は、同じラベルのまま変換例を増やし、モデルへ不変性を教える。

ただし変換が強すぎるとラベルを壊す。このデータの回転は**必ず±25度以内**。
それ以上では回転した bottle が横長の book と同じ形に近づき、クラス定義自体が曖昧になることを実測済み。

## ② 解説: trainだけをランダムに変え、validは固定する

| 拡張 | ラベルを保ちやすい条件 | 壊れる例 |
|---|---|---|
| 左右反転 | 左右で意味が変わらない商品 | 文字・左右専用品 |
| 回転 | **この教材では±25度以内** | 大回転で bottle と book の形が衝突 |
| 明度/コントラスト | 物体が見える範囲 | 白飛び・黒潰れ |
| ノイズ | 軽いセンサノイズ | 形が消えるほど強いノイズ |
| crop | 対象が残る | 商品本体を切り落とす |

拡張は **training fold の `__getitem__` だけ**でランダム適用し、valid / test は決定的な resize・normalize だけ。
valid まで毎回変わると、指標の変化がモデル改善なのか乱数なのか分からない。

誤分類分析ではスコアだけで終わらず、`image_id / true / predicted / product_key` を元表へ戻す。
同じ商品群がまとめて失敗しているなら、遮蔽・回転・背景など共通の nuisance factor を画像で確認する。

In [ ]:
# GOAL: ラベルを保つ4変換を、shape・dtype・値域を壊さず適用する

aug_source = image_hwc
rng_aug = np.random.default_rng(8)
flipped = np.flip(aug_source, axis=1).copy()
bright = np.clip(aug_source.astype(np.float32) * 1.15, 0, 255).astype(np.uint8)
noisy = np.clip(aug_source.astype(np.float32) + rng_aug.normal(0, 6, aug_source.shape), 0, 255).astype(np.uint8)
rotated = np.asarray(Image.fromarray(aug_source).rotate(
    25, resample=Image.Resampling.BILINEAR, fillcolor=(240, 240, 240)))

for name, image in [("source", aug_source), ("flip", flipped), ("bright", bright),
                    ("noise", noisy), ("rotate +25°", rotated)]:
    print(f"{name:<12} shape={image.shape} dtype={image.dtype} range={image.min()}-{image.max()}")
print("回転は +25度。生成データと教材コードの上限は常に±25度。")

## ④ 予測: 拡張を強くすれば、いつでも強くなる?

次のセルでは、検証済みの精度表を正本の固定値として見る。

1. 作り込み不足の scratch CNN 0.5731 だけを見て「CNNはHOGより弱い」と結論してよい?
2. BatchNorm・拡張・cosine schedule を入れた scratch CNN はどこまで上がる?
3. 小データで全層fine-tuneより凍結headが勝つ理由は?

アーキテクチャの優劣を語る前に、学習レシピと分割条件が揃っているかを確認する。

In [ ]:
# GOAL: 「CNNがHOGに負ける」という古い結論を、訂正後の実測表で更新する

verified_scores = pd.DataFrame({
    "method": [
        "色hist + LinearSVC", "HOG + LinearSVC", "scratch CNN(作り込み不足)",
        "scratch CNN(BN+拡張+cosine)", "転移学習(全層FT)", "転移学習(凍結head)",
    ],
    "accuracy": [0.3533, 0.7840, 0.5731, 0.9038, 0.9451, 0.9538],
})
print(verified_scores.to_string(index=False))
print("\n重要な訂正:")
print("  0.5731はCNNの限界ではなく、BatchNorm・拡張・schedule不足。作り込むと0.9038。")
print("  転移学習は作り込んだscratchにも +0.050。")
print("  小データでは凍結0.9538 > 全層FT0.9451。まずheadだけ、足りなければ段階的に解凍。")
print("  回転は±25度以内。強い拡張でラベル定義を壊してはいけない。")

print("\nHOGの誤分類クラス別件数:")
print(errors.groupby(["label", "predicted"]).size().sort_values(ascending=False).head(8))
print("スコア差の次は、errors の image_id を開き、遮蔽・背景・回転を仮説として分類する。")

## ⑥ 書いてみる: 決定的な flip + brightness 拡張

`augment_flip_brightness(image, flip, brightness)` を完成させよう。

- float32 のコピーを作る
- `flip=True` なら `np.flip(..., axis=1)` で左右反転
- brightness を掛け、`np.clip(..., 0.0, 1.0)`
- 元画像と同じ shape の float32 を返し、入力を破壊しない

4〜5行。実務のランダム性は外から `rng` で決めると、seed固定で再現できる。

In [ ]:
# STEP 20: ⑥ 書いてみる: 決定的な flip + brightness 拡張の処理を実行し、出力を照合する
def augment_flip_brightness(image, flip, brightness):
    # ここに書く(ヒント: float32のcopy→必要ならaxis=1反転→brightness→clip)
    return None


aug_e_input = np.arange(12, dtype=np.float32).reshape(2, 2, 3) / 11.0
aug_e = call_safely(augment_flip_brightness, aug_e_input, True, 0.5)
print("aug_e:", shape_safely(aug_e), None if aug_e is None else aug_e.dtype)

In [ ]:
# STEP 21: ⑥ 書いてみる: 決定的な flip + brightness 拡張の処理を実行し、出力を照合する
# ===== チェックポイント E: ラベル保存拡張 =====
check("E-1 shape", shape_safely(aug_e), (2, 2, 3),
      hint="左右反転と明度変更はshapeを変えない。")
check("E-2 dtype", str(aug_e.dtype) if aug_e is not None else None, "float32",
      hint="astype(np.float32).copy() から始める。")
check("E-3 値", aug_e, [[[0.13636364042758942, 0.1818181872367859, 0.22727273404598236], [0.0, 0.04545454680919647, 0.09090909361839294]], [[0.40909090638160706, 0.4545454680919647, 0.5], [0.27272728085517883, 0.3181818127632141, 0.3636363744735718]]],
      hint="axis=1で左右を入れ替え、その後0.5倍。")
_e_input_value = float(aug_e_input[0, 0, 0]) if aug_e is not None else None
check("E-4 入力を破壊しない", _e_input_value, 0.0,
      hint="入力へin-place乗算せず、copyしたoutを変換する。")

<!-- DHASH_FOLD_SECTION_UNIT08 -->
## aHashの次: dHash と fold 内 augmentation

dHash（difference hash）は平均との比較ではなく、横に隣り合う画素の明暗を比較します。幅を1画素多い `(hash_size + 1, hash_size)` へ縮小すると、左側と右側を同じshapeで比較できます。

augmentation は **splitした後のtrain indexだけ**へ適用します。先に全画像を拡張すると、同じ元画像の派生版がvalidへ入り、unit02と同じ重複リークになります。

In [ ]:
# STEP 1: aHashとdHashを同じ64 bitで比較する
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

dhash_source = np.tile(np.arange(9, dtype=np.uint8) * 20, (8, 1))
dhash_resized = Image.fromarray(dhash_source).resize((9, 8), Image.Resampling.LANCZOS)
dhash_values = np.asarray(dhash_resized, dtype=float)
dhash_bits = (dhash_values[:, 1:] >= dhash_values[:, :-1]).reshape(-1)
ahash_values = np.asarray(Image.fromarray(dhash_source).resize((8, 8)), dtype=float)
ahash_bits = (ahash_values >= ahash_values.mean()).reshape(-1)
print("aHash / dHash shape:", ahash_bits.shape, dhash_bits.shape)
print("Hamming distance:", int(np.not_equal(ahash_bits, dhash_bits).sum()))

# STEP 2: group split後、各foldのtrain indexだけを拡張対象へ記録する
fold_labels = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1])
fold_groups = np.array([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5])
fold_splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
augmentation_boundaries = []
for train_index, valid_index in fold_splitter.split(np.zeros(len(fold_labels)), fold_labels, fold_groups):
    augmented_train_index = train_index.copy()  # 実処理ではこの画像だけを反転・明度変更する
    augmentation_boundaries.append((augmented_train_index, valid_index))
    print("train/valid group overlap:", set(fold_groups[train_index]) & set(fold_groups[valid_index]))
print("fold数:", len(augmentation_boundaries))

---
## 振り返り(自己評価 + TIL)

以下に1〜2文ずつ、自分の言葉で書いてみよう。

**1. 今日学んだこと:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック:**

- HWC画像のチャネル標準化で `axis=(0,1), keepdims=True` にする理由をshapeで説明できる?
- 完全一致3/378なのに、正規化thumbnail cosineが役立つ理由を説明できる?
- 同一商品の写真を train / valid に分けてはいけない理由を、リークの言葉で説明できる?

> (ここに書く)

---
## まとめ

| 概念 | 一言でいうと |
|---|---|
| HWC / CHW | Pillow系は `(H,W,C)`、PyTorchは `(C,H,W)`。最初にshapeをprint |
| RGB / BGR | OpenCVだけBGR。shapeが同じでも意味が違う |
| uint8 / float32 | 読込は0〜255、演算は明示的にfloat32へ。uint8演算はoverflow |
| resize / normalize | サイズと明るさをそろえる。mean/stdのshapeはHWCなら `(1,1,3)` |
| 近傍重複 | 完全一致は3/378。知覚hashや正規化縮小画像で見た目を比べる |
| 色ヒストグラム | 位置を捨て色分布を残す。この5クラスではaccuracy 0.3533 |
| HOG / Sobel | 輪郭・勾配方向を残す。HOG+LinearSVCは0.7840 |
| group-aware CV | `StratifiedGroupKFold(groups=product_key)` で同一商品をfold間分離 |
| 誤分類分析 | OOF予測を image_id へ戻し、遮蔽・背景・回転の仮説を画像で確認 |
| augmentation | 学習foldだけへラベル保存変換。回転は**必ず±25度以内** |

### この先どこで使うか

- **unit09** — HOG 0.7840を基準に、作り込んだscratch CNN 0.9038、転移学習0.9538へ進む。
  0.5731は作り込み不足の結果であり、「CNNがHOGより弱い」という結論には使わない。
- **unit10** — HOG+線形の軽いCPU推論とCNNの精度・コストを比較する。
- **実務** — 商品画像の重複排除、group leakage防止、低コスト画像分類へそのまま使う。

**次は演習 `ex01_image_array_ops` へ進もう。lesson.ipynb を見ながらで OK。**

| 演習 | 内容 |
|---|---|
| `ex01_image_array_ops` | HWC/CHW、RGB/BGR、dtype、grayscale |
| `ex02_resize_and_normalize` | resize、aspect ratio、channel標準化 |
| `ex03_perceptual_hash_dedup` | aHash/dHash、ハミング距離、重複閾値 |
| `ex04_capstone` | HOG特徴、group CV、誤分類分析、augmentation |